# Reproducible Analyses with DataLad Run

**Author:** Sin Kim, Michèle Masson-Trottier

**Date:** 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

**Note:** If this notebook uses neuroimaging tools from Neurocontainers, those tools retain their original licenses. Please see <a href="https://neurodesk.org/overview/how-to-cite-us/" target="_blank" style="color: #0066cc;">Neurodesk citation guidelines</a> for details.

### Citation and Resources:

#### Tools included in this workflow
- **DataLad** - Halchenko, Y., et al. (2021). DataLad: distributed system for joint management of code, data, and their relationship. *Journal of Open Source Software*, 6(63), 3262. https://doi.org/10.21105/joss.03262

#### Educational resources
- [DataLad Handbook](http://handbook.datalad.org/en/latest/)
- [YODA principles](http://handbook.datalad.org/en/latest/basics/101-127-yoda.html)
- [Brief terminal guide](http://handbook.datalad.org/en/latest/intro/howto.html)
- [datalad-container extension](http://handbook.datalad.org/en/latest/basics/101-133-containersrun.html)

## Load software tools

In [ ]:
import module
await module.load('datalad/0.19.6')
await module.load('julia/1.8.5')
await module.list()

## 1. Create a YODA-Structured DataLad Project

YODA is a set of conventions for organizing DataLad datasets for reproducible analysis. The `-c yoda` flag applies these conventions automatically:

In [ ]:
%%bash
cd ~/neurodesktop-storage/
datalad create -c yoda SomeProject
cd SomeProject
ls

The YODA structure separates **code** from **outputs** and tracks both with full provenance.

In [ ]:
%%bash
cd ~/neurodesktop-storage/SomeProject
tree . 2>/dev/null || ls -la

## 2. Create an Analysis Script

Write a simple Julia script that produces an output. We use a heredoc to create the file from bash:

In [ ]:
%%bash
cd ~/neurodesktop-storage/SomeProject
cat > code/hello.jl << 'EOF'
println("hello neurodesktop")
EOF
cat code/hello.jl

## 3. Test the Script Runs Correctly

In [ ]:
%%bash
cd ~/neurodesktop-storage/SomeProject
julia code/hello.jl

## 4. Record the Run with DataLad

Save the script first, then use `datalad run` to execute it. DataLad records the exact command, inputs, and outputs in the commit history:

In [ ]:
%%bash
cd ~/neurodesktop-storage/SomeProject
datalad save -m "Add hello script" code/
mkdir -p outputs
datalad run \
  -m "run hello script" \
  -o "outputs/hello.txt" \
  "julia code/hello.jl > outputs/hello.txt"
cat outputs/hello.txt

**What `datalad run` records:**
- `-m` - human-readable message for the commit
- `-o` - output files to track (cleaned before re-running)
- The full command is stored in the commit metadata

## 5. Inspect the Provenance Record

View the recorded commit history to see the full provenance:

In [ ]:
%%bash
cd ~/neurodesktop-storage/SomeProject
git log --oneline -5

In [ ]:
%%bash
cd ~/neurodesktop-storage/SomeProject
git log -1 --format="%B" outputs/hello.txt

## 6. Re-run the Analysis

Anyone who clones this dataset can reproduce the exact results using `datalad rerun`:

In [ ]:
%%bash
cd ~/neurodesktop-storage/
datalad install -s ~/neurodesktop-storage/SomeProject ReproducedProject
cd ReproducedProject
COMMIT=$(git log --format="%H" -1)
datalad rerun $COMMIT
cat outputs/hello.txt

## 7. Verify Reproducibility

In [ ]:
%%bash
cd ~/neurodesktop-storage/ReproducedProject
diff outputs/hello.txt ~/neurodesktop-storage/SomeProject/outputs/hello.txt && echo "✓ Outputs are identical"

## Dependencies in Jupyter/Python
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions